In [ ]:
!pip install transformers datasets torch scikit-learn pandas iterative-stratification

In [ ]:
import inspect
import pandas as pd
import numpy as np
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import f1_score
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

In [ ]:
# 설정
MODEL_NAME = "searle-j/kote_for_easygoing_people"
MAX_LEN = 128
THRESHOLD = 0.4

# 감정_대분류 → KOTE 44 감정 매핑
label_map = {
    "기쁨": ["기쁨"],
    "당황": ["당황/난처"],
    "분노": ["화남/분노"],
    "불안": ["불안/걱정"],
    "상처": ["서러움"],
    "슬픔": ["슬픔"],
}

df = pd.read_csv("aihub_emotion.csv")
texts = df["사람문장1"].tolist()
raw_labels = df["감정_대분류"].tolist()

In [ ]:
# multi-hot vector
kote_labels = [
    "불평/불만", "환영/호의", "감동/감탄", "지긋지긋", "고마움",
    "슬픔", "화남/분노", "존경", "기대감", "우쭐댐/무시함",
    "안타까움/실망", "비장함", "의심/불신", "뿌듯함", "편안/쾌적",
    "신기함/관심", "아껴주는", "부끄러움", "공포/무서움", "절망",
    "한심함", "역겨움/징그러움", "짜증", "어이없음", "없음",
    "패배/자기혐오", "귀찮음", "힘듦/지침", "즐거움/신남", "깨달음",
    "죄책감", "증오/혐오", "흐뭇함(귀여움/예쁨)", "당황/난처", "경악",
    "부담/안_내킴", "서러움", "재미없음", "불쌍함/연민", "놀람",
    "행복", "불안/걱정", "기쁨", "안심/신뢰",
]
assert len(kote_labels) == 44

label_to_idx = {label: i for i, label in enumerate(kote_labels)}

def encode_label(category):
    vec = np.zeros(len(kote_labels))
    for m in label_map.get(category, []):
        if m in label_to_idx:
            vec[label_to_idx[m]] = 1
    return vec


labels = np.array([encode_label(cat) for cat in raw_labels], dtype=np.float32)

In [ ]:
# train / validation / test split (80 / 10 / 10)
texts_arr = np.array(texts)

# 1차 split: train 80%, temp 20%
msss = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, temp_idx = next(msss.split(texts_arr, labels))

train_texts = texts_arr[train_idx].tolist()
train_labels = labels[train_idx]

temp_texts = texts_arr[temp_idx]
temp_labels = labels[temp_idx]


# 2차 split: validation 10%, test 10%
msss_test = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.5,
    random_state=42
)

val_idx, test_idx = next(
    msss_test.split(temp_texts, temp_labels)
)

val_texts = temp_texts[val_idx].tolist()
val_labels = temp_labels[val_idx]

test_texts = temp_texts[test_idx].tolist()
test_labels = temp_labels[test_idx]


train_dataset = Dataset.from_dict({
    "text": train_texts,
    "labels": train_labels
})

val_dataset = Dataset.from_dict({
    "text": val_texts,
    "labels": val_labels
})

test_dataset = Dataset.from_dict({
    "text": test_texts,
    "labels": test_labels
})

In [ ]:
# 토큰화
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN
    )

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

# 불필요 컬럼 제거
train_dataset = train_dataset.remove_columns(["text"])
val_dataset = val_dataset.remove_columns(["text"])
test_dataset = test_dataset.remove_columns(["text"])

In [ ]:
# 모델 로드
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    problem_type="multi_label_classification",
)

original_order = [model.config.id2label[i] for i in range(44)]

# warning 처리
if original_order != kote_labels:
    print("⚠️ label 순서 불일치 감지 (학습은 진행됨)")
    print("checkpoint:", original_order)
    print("current   :", kote_labels)

model.config.id2label = {i: label for i, label in enumerate(kote_labels)}
model.config.label2id = {label: i for i, label in enumerate(kote_labels)}

In [ ]:
mapped_indices = sorted({
    label_to_idx[l]
    for vals in label_map.values()
    for l in vals
})

loss_mask = torch.zeros(len(kote_labels))
loss_mask[mapped_indices] = 1.0

In [ ]:
class MaskedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        raw_loss = torch.nn.functional.binary_cross_entropy_with_logits(
            logits, labels,
            reduction="none"
        )

        mask = loss_mask.to(logits.device)
        loss = (raw_loss * mask).sum() / mask.sum()

        return (loss, outputs) if return_outputs else loss

In [ ]:
# 평가 함수
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    # numpy / torch 혼재 방어
    if isinstance(logits, torch.Tensor):
        logits = logits.detach().cpu().numpy()

    probs = 1 / (1 + np.exp(-logits))
    preds = (probs > THRESHOLD).astype(int)

    return {
        "f1_micro": f1_score(labels, preds, average="micro"),
        "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),
    }

# transformers 버전에 따라 evaluation_strategy / eval_strategy 자동 선택
eval_strategy_kwarg = (
    "eval_strategy"
    if "eval_strategy" in inspect.signature(TrainingArguments.__init__).parameters
    else "evaluation_strategy"
)

In [ ]:
# 학습 설정
training_args = TrainingArguments(
    output_dir="./kote_finetuned",

    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,

    **{eval_strategy_kwarg: "epoch"},  # 수정
    save_strategy="epoch",

    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1_macro",

    report_to="none"
)

# 학습
trainer = MaskedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

print(
    "Best checkpoint:",
    trainer.state.best_model_checkpoint
)

print(
    "Best metric:",
    trainer.state.best_metric
)

# 최종 test 평가
test_result = trainer.evaluate(
    eval_dataset=test_dataset
)

print("===== TEST RESULT =====")
print(test_result)

# 저장
model.save_pretrained("./kote_finetuned")
tokenizer.save_pretrained("./kote_finetuned")
